# Gemma4GR — Phase 2: Greek Q&A LoRA (E4B) on Colab

**GPU:** L4 (40 GB) or A100 (80 GB) — set via Runtime → Change runtime type

**What this does:** Fine-tunes Gemma 4 E4B on Greek Q&A pairs using text-only QLoRA.  
Saves the LoRA adapter to Google Drive, ready for merging on your local machine.

**Steps:**
1. Set your Colab Secrets (HF_TOKEN)
2. Mount Google Drive (your Q&A data must be uploaded there first)
3. Run all cells in order
4. Download the `lora_adapter/` folder from Google Drive

In [ ]:
# ── Step 1: Load secrets ─────────────────────────────────────────────────────
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')   # Add via 🔑 icon in the left sidebar
print('HF_TOKEN loaded:', HF_TOKEN[:8] + '...' if HF_TOKEN else 'MISSING — add it in Colab Secrets')

In [ ]:
# ── Step 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Expected path: Google Drive/Gemma4GR/data/train_qa.jsonl
import os
DRIVE_ROOT = '/content/drive/MyDrive/Gemma4GR'
DATA_DIR   = f'{DRIVE_ROOT}/data'
OUTPUT_DIR = f'{DRIVE_ROOT}/output/e4b_greek_qa'

os.makedirs(OUTPUT_DIR, exist_ok=True)

train_file = f'{DATA_DIR}/train_qa.jsonl'
val_file   = f'{DATA_DIR}/val_qa.jsonl'

if not os.path.exists(train_file):
    print(f'ERROR: {train_file} not found.')
    print('Upload your data/train_qa.jsonl and data/val_qa.jsonl to Google Drive/Gemma4GR/data/')
else:
    with open(train_file) as f:
        n = sum(1 for _ in f)
    print(f'train_qa.jsonl: {n} examples ✓')

In [ ]:
# ── Step 3: Install Unsloth ───────────────────────────────────────────────────
%%capture
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install',
    'unsloth[colab-new]', 'trl>=0.9.0', 'datasets', 'python-dotenv',
    '--quiet'
], check=True)

print('Unsloth + TRL installed ✓')

In [ ]:
# ── Step 4: Verify GPU ────────────────────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1e9
    print(f'GPU:  {props.name} — {vram:.1f} GB VRAM')
    if vram < 20:
        print('WARNING: E4B QA training needs 20+ GB. Upgrade to L4 or A100.')

In [ ]:
# ── Step 5: Train ─────────────────────────────────────────────────────────────
import os, json, torch
from datetime import datetime
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN

MODEL_NAME   = 'unsloth/gemma-4-E4B-it'
MAX_SEQ_LEN  = 2048
LORA_R       = 32
LORA_ALPHA   = 64
EPOCHS       = 3
BATCH_SIZE   = 2
GRAD_ACCUM   = 4
LR           = 1e-4

print(f'Loading {MODEL_NAME} ...')
model, processor = FastModel.from_pretrained(
    model_name     = MODEL_NAME,
    dtype          = None,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
    full_finetuning= False,
    token          = HF_TOKEN,
)
print('Model loaded ✓')

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers    = False,
    finetune_language_layers  = True,
    finetune_attention_modules= True,
    finetune_mlp_modules      = True,
    r            = LORA_R,
    lora_alpha   = LORA_ALPHA,
    lora_dropout = 0,
    bias         = 'none',
)

tokenizer = get_chat_template(processor.tokenizer, chat_template='gemma-4')
processor.tokenizer = tokenizer

train_ds = load_dataset('json', data_files=train_file, split='train')
val_ds   = load_dataset('json', data_files=val_file,   split='train')
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

use_bf16 = torch.cuda.is_bf16_supported()
run_dir  = f"{OUTPUT_DIR}/{datetime.now().strftime('%Y%m%d_%H%M')}"
os.makedirs(run_dir, exist_ok=True)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = processor.tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    args = SFTConfig(
        output_dir                  = run_dir,
        dataset_text_field          = 'text',
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = 10,
        num_train_epochs            = EPOCHS,
        learning_rate               = LR,
        logging_steps               = 5,
        eval_steps                  = 50,
        save_steps                  = 200,
        save_total_limit            = 2,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.001,
        lr_scheduler_type           = 'cosine',
        fp16                        = not use_bf16,
        bf16                        = use_bf16,
        use_gradient_checkpointing  = 'unsloth',
        report_to                   = 'none',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        greater_is_better           = False,
        max_seq_length              = MAX_SEQ_LEN,
    ),
)

print('Starting training ...')
stats = trainer.train()
print(f'Final loss: {stats.training_loss:.4f}')

In [ ]:
# ── Step 6: Save adapter to Drive ─────────────────────────────────────────────
adapter_dir = f'{OUTPUT_DIR}/lora_adapter'
model.save_pretrained(adapter_dir)
processor.tokenizer.save_pretrained(adapter_dir)

print(f'Adapter saved to Google Drive: {adapter_dir}')
print()
print('Next steps:')
print('  1. Download the lora_adapter/ folder from Google Drive')
print('  2. Copy it to your local: output/e4b_greek_qa/lora_adapter/')
print('  3. Run step G (merge_adapters.py) on your local machine')

In [ ]:
# ── Step 7 (Optional): Quick inference test ───────────────────────────────────
FastModel.for_inference(model)

test_question = 'Ποια είναι η πρωτεύουσα της Ελλάδας;'

messages = [{"role": "user", "content": test_question}]
inputs = processor.tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
).to('cuda')

outputs = model.generate(
    input_ids=inputs, max_new_tokens=200, temperature=0.7,
    do_sample=True, use_cache=True,
)
response = processor.tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(f'Q: {test_question}')
print(f'A: {response}')